# Fraudulent Transactions — Modeling

## Methodology: CRISP-DM

This notebook covers phases 5 through 9. Phases 1 to 4 are in `01_eda.ipynb`.

5. Data cleaning and preprocessing
6. Model training, comparison, selection and tuning
7. Final production model testing and evaluation
8. Conclude and interpret the model results
9. Deploy

All logic lives in the `fraud_detection` package. This notebook is the narrative and
the call sequence; nothing is defined here.

In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt

from fraud_detection import config as cfg
from fraud_detection import preprocessing as prep
from fraud_detection import model as mdl
from fraud_detection import reporting as rpt
from fraud_detection import deployment as dep

# **5. DATA AND PARTITIONS**

In [ ]:
df = prep.load_transactions()

splits = prep.chronological_split(df)
summary = prep.describe_splits(splits)
summary

In [ ]:
rpt.show(rpt.split_insights(summary))

#### **Preprocessing contract**

`prep.build_pipeline` returns preprocessing, optional resampling and the estimator as a
single object. Resampling is a pipeline *step*, so it is re-fitted on every training fold
and never sees the validation fold — applying SMOTE to a dataset in advance is the leak
that signature exists to prevent. Note that `sklearn.pipeline.Pipeline` rejects
resamplers, hence imblearn's variant.

Two feature sets are declared in `config`. `REALTIME_FEATURES` holds only what exists at
authorisation time. `FORENSIC_FEATURES` adds four post-settlement balances which, in
PaySim, encode the label almost tautologically — valid for a post-settlement review
queue, never for real-time scoring. The gap between the two is the measured cost of that
leakage.

# **6. MODEL TRAINING, COMPARISON, SELECTION AND TUNING**

In [ ]:
search = prep.stratified_search_sample(splits.train)
print(f"Search sample: {search.height:,} rows | "
      f"{search[cfg.TARGET].sum():,} frauds ({search[cfg.TARGET].mean():.4%})")

In [ ]:
df_results = mdl.screen_models(search, cfg.FORENSIC_FEATURES)
rpt.plot_screening(df_results)
plt.show()
df_results

In [ ]:
rpt.show(rpt.screening_insights(df_results, search))

In [ ]:
studies = mdl.tune(search)

In [ ]:
rpt.plot_tuning(studies)
plt.show()

In [ ]:
rpt.show(rpt.tuning_insights(studies, df_results, search))

# **7. FINAL PRODUCTION MODEL TESTING AND EVALUATION**

In [ ]:
results = mdl.evaluate_feature_sets(splits, studies)
mdl.summarize(results)

In [ ]:
rpt.show(rpt.evaluation_insights(results))

# **8. CONCLUDE AND INTERPRET THE MODEL RESULTS**

In [ ]:
deployable = results['realtime']

importance = mdl.gain_importance(deployable.pipeline)
rpt.plot_importance(importance, title='Feature importance — real-time model (gain)')
plt.show()
importance.to_frame('gain')

In [ ]:
mdl.permutation_table(deployable.pipeline, splits.val, cfg.REALTIME_FEATURES)

In [ ]:
rpt.show(rpt.conclusion_insights(results, importance))

##### 3. Honest Limitations
* **Synthetic data.** PaySim is a simulator. Its fraud agents follow programmed rules, which are learnable in a way real adversaries are not. Every number here is an upper bound on real-world performance.
* **No adversarial drift.** The model is validated on one month of a static simulation. Real fraud patterns shift in response to detection, so production performance decays without retraining.
* **Single-transaction scope.** The model scores each transaction in isolation. It cannot see that an account received three transfers in the previous ten minutes, which is how fraud rings actually appear.
* **Threshold assumes symmetric costs.** F1 optimisation treats a blocked legitimate customer and a missed fraud as equally bad. They are not.
* **Train and test are not exchangeable.** Fraud prevalence is several times higher in the test period than in training, so the reported score describes a more hostile regime than the model learned from. Realistic, but not a stationary estimate.

##### 4. Recommended Next Steps
* Engineer per-account velocity aggregates over a trailing window of `step`, computed strictly from past rows to preserve causal ordering.
* Re-derive the threshold with `mdl.threshold_for_cost_ratio` once the business supplies a false-negative to false-positive cost ratio.
* Establish a monitoring baseline on prediction distribution and flag rate, so drift is detected before recall degrades silently.
* Validate against real transactional data before any production consideration.

# **9. DEPLOY**

In [ ]:
artifact_path = dep.save_artifact(deployable, splits, studies['realtime'])

In [ ]:
preview = dep.score_transactions(splits.test.head(10_000), dep.load_artifact(artifact_path))
print(f"Flagged: {preview['flagged'].sum()} of {preview.height}")
preview.select(['step', 'type', 'amount', 'isFraud', 'fraud_probability', 'flagged']).head()

#### **Scoring contract**

`dep.score_transactions(frame, artifact)` requires every column in
`artifact['features']`. Three are derived rather than raw — `is_merchant_dest`,
`hour_of_day`, `day_of_week` — and are the caller's responsibility; all three depend only
on fields available at authorisation time.

Post-settlement columns are **rejected, not ignored**. A caller supplying them is
describing a transaction that has already completed, at which point prevention is no
longer possible, and silently accepting them is how a forensic model ends up deployed as
a real-time one.

The threshold ships inside the artifact deliberately. A model saved without its operating
point is not usable: whoever loads it will call `predict` at 0.5 and get a recall
unrelated to the one reported above.